# Orbit ML Workspace

Clean working notebook for the current message-passing GNN pipeline.

This notebook intentionally excludes legacy flat `ginputs` / `gnn_targets` work, old `GNN` imports, and exploratory cells from the pre-refactor model. The active pipeline is graph-first:

`raw graph snapshots -> ScalerBundle -> MPNN.forward -> unscaled acceleration predictions`

Use this notebook for data readiness checks, graph-snapshot construction, smoke tests, and quick post-training inspection.


## 1. Project Setup

Run these cells from a fresh kernel. If the notebook is opened from `notebooks/`, the setup cell moves the process to the repository root so all project-relative paths match the scripts.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

# Keep notebook execution self-contained on systems where user cache directories are not writable.
NOTEBOOK_CACHE_DIR = Path("/private/tmp/orbit_ml_notebook_cache")
NOTEBOOK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(NOTEBOOK_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(NOTEBOOK_CACHE_DIR / "xdg"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import shutil
import time
from contextlib import contextmanager

from IPython.display import display

import numpy as np
import torch

import config
from data_construction import NNDataModule
from models import MPNN
from integration import gnn_force, newton_force
from utils import (
    ScalerBundle,
    load_checkpoint_for_model,
    build_graph_snapshots,
    calc_mae,
    calc_mape,
    fully_connected_edges,
    gnn_test,
    plot_pareto,
    print_best_optuna,
    print_block,
)

# Keep notebook dtype aligned with the project config.
torch.set_default_dtype(torch.float64) if not config.MAC else torch.set_default_dtype(torch.float32)

print_block("Active Configuration")
print(f"config.TYPE: {config.TYPE}")
print(f"torch default dtype: {torch.get_default_dtype()}")
print(f"graph file: {config.GRAPH_FILE}")
print(f"scaler file: {config.SCALER_FILE}")


## 2. Reusable Inspection Helpers

Small notebook-local wrappers around project utilities. These keep later cells focused on scientific checks instead of repeated path and shape plumbing.


In [ ]:
def file_size(path):
    path = Path(path)
    if not path.exists():
        return "missing"
    size = path.stat().st_size
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024
    return f"{size:.1f} PB"


def finite_all(arr, chunk_size=200_000):
    for start in range(0, arr.shape[0], chunk_size):
        stop = min(start + chunk_size, arr.shape[0])
        if not np.isfinite(arr[start:stop]).all():
            return False
    return True


def describe_numpy(name, arr):
    arr = np.asarray(arr)
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")
    print(f"  finite={np.isfinite(arr).all()}")
    print(f"  min={np.nanmin(arr):.6g}, mean={np.nanmean(arr):.6g}, max={np.nanmax(arr):.6g}")



SECONDS_PER_JULIAN_YEAR = 365.25 * 24 * 60 * 60
KM_TO_M = 1000.0


HORIZONS_UNITS = {
    "JDTDB": "Julian day (TDB)",
    "X/Y/Z": "km in the source CSV, converted to m by load_np",
    "VX/VY/VZ": "km/s in the source CSV, converted to m/s by load_np",
}


GRAPH_UNITS = {
    "mass": "kg",
    "x/y/z": "m",
    "vx/vy/vz": "m/s",
    "ax/ay/az": "m/s^2",
}


def mercury_newton_accel(r_vec):
    r_vec = np.asarray(r_vec, dtype=np.float64)
    return -config.GM_SUN * r_vec / np.linalg.norm(r_vec) ** 3


def graph_physical_summary(graph, idx=0):
    r_vec = np.asarray(graph[idx, 1, 1:4])
    v_vec = np.asarray(graph[idx, 1, 4:7])
    a_target = np.asarray(graph[idx, 1, 7:10])
    a_newton = mercury_newton_accel(r_vec)
    ratio = np.linalg.norm(a_target) / np.linalg.norm(a_newton)
    return {
        "idx": idx,
        "r_norm_m": np.linalg.norm(r_vec),
        "v_norm_m_per_s": np.linalg.norm(v_vec),
        "a_target_norm_m_per_s2": np.linalg.norm(a_target),
        "a_newton_norm_m_per_s2": np.linalg.norm(a_newton),
        "target_to_newton_ratio": ratio,
    }


def print_physical_summary(graph, idx=0):
    summary = graph_physical_summary(graph, idx=idx)
    print_block(f"Physical Units Check: graph index {idx}")
    for key, value in summary.items():
        if key == "idx":
            print(f"{key}: {value}")
        else:
            print(f"{key}: {value:.8g}")
    return summary


def inspect_graph_file(path=config.GRAPH_FILE):
    graph = np.load(path, mmap_mode="r")
    print_block(f"Graph Artifact: {path}")
    print(f"shape: {graph.shape}")
    print(f"dtype: {graph.dtype}")
    print(f"finite: {finite_all(graph)}")
    print(f"first snapshot masses: {np.asarray(graph[0, :, 0])}")
    print(f"sun non-mass max abs: {np.max(np.abs(graph[:, 0, 1:])):.6g}")
    print(f"all-body target std: {np.std(graph[..., 7:10], axis=(0, 1))}")
    if graph.shape[1] > 1:
        print(f"Mercury target std: {np.std(graph[:, 1, 7:10], axis=0)}")
    return graph


def checkpoint_candidates():
    roots = [Path("tlogs/checkpoints"), Path("hopt/checkpoints"), Path(".")]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.glob("*.ckpt"))
    return sorted(candidates, key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)


@contextmanager
def pushd(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)

CURRENT_GNN_PARAM_KEYS = {"edge_hidden_dim", "edge_layers", "node_hidden_dim", "node_layers", "msg_dim", "msg_l1"}


def checkpoint_architecture(path):
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        state = ckpt.get("state_dict", ckpt)
        keys = set(state.keys())
    except Exception as exc:
        return "unreadable", str(exc)

    if any(k.startswith("edge_model.") for k in keys) and any(k.startswith("node_model.") for k in keys):
        return "mpnn", "current message-passing architecture"
    if any(k.startswith("input_layer.") for k in keys) or any(k.startswith("mlp_layers.") for k in keys):
        return "legacy_flat", "legacy SEMLP/flat-MLP style checkpoint"
    return "unknown", "state_dict keys do not match known active MPNN layout"


def compatible_mpnn_checkpoints():
    compatible = []
    for path in checkpoint_candidates():
        arch, detail = checkpoint_architecture(path)
        print(f"{path}: {arch} ({detail})")
        if arch == "mpnn":
            compatible.append(path)
    return compatible


def load_current_mpnn_checkpoint(path):
    arch, detail = checkpoint_architecture(path)
    if arch != "mpnn":
        raise ValueError(
            f"{path} is not a current MPNN checkpoint: {arch} ({detail}). "
            "Do not load legacy flat checkpoints with MPNN.load_from_checkpoint."
        )
    return load_checkpoint_for_model(MPNN, path)


def study_looks_current_gnn(study):
    trials = [t for t in study.trials if t.params]
    if not trials:
        return False
    param_keys = set().union(*(set(t.params) for t in trials))
    return bool(CURRENT_GNN_PARAM_KEYS & param_keys)


## 3. Data And Artifact Inventory

Use this before training to decide whether data generation is needed. For the default Sun + Mercury run, `data/horizons.csv` is enough to build `data/graph_data.npy`; a fresh Horizons fetch is only needed if changing the date range, cadence, or body set.


In [ ]:
print_block("Data And Artifact Inventory")

important_paths = [
    Path(config.DATA_DIR) / "horizons.csv",
    Path(config.DATA_DIR) / "earth_pos.csv",
    Path(config.GRAPH_FILE),
    Path(config.SCALER_FILE),
    Path("tlogs/checkpoints/gnn_final.ckpt"),
    Path(config.MPNN_CHECKPOINT_FILE),
    Path("pysr/distilled.pkl"),
]

for p in important_paths:
    print(f"{str(p):38s} {file_size(p)}")

ckpts = checkpoint_candidates()
if ckpts:
    print("\nRecent checkpoints:")
    for p in ckpts[:5]:
        print(f"{str(p):38s} {file_size(p)}")
else:
    print("\nNo local checkpoints found.")


In [ ]:
graph_path = Path(config.GRAPH_FILE)
scaler_path = Path(config.SCALER_FILE)

if not graph_path.exists() and scaler_path.exists():
    print("WARNING: graph_data.npy is missing but a GNN scaler already exists.")
    print("Archive or delete the scaler before fitting a new datamodule/training run,")
    print("otherwise later model.predict calls may load stale scaler statistics.")
else:
    print("No graph/scaler mismatch detected by this quick check.")


## 4. Units Ledger And Source Sanity Checks

Keep the unit story explicit before training:

- Horizons CSV: `JDTDB` is Julian days, positions are km, velocities are km/s.
- `utils.data_processing.load_np`: converts `JDTDB` to decimal Julian years, positions to m, velocities to m/s.
- `utils.numerical_methods.get_movements`: differentiates with respect to decimal years, then divides by seconds per Julian year so accelerations are m/s^2.
- `data/graph_data.npy`: stores `[mass, x, y, z, vx, vy, vz, ax, ay, az]` as `[kg, m, m/s, m/s^2]`.
- `NNDataModule`: fits standard scalers over graph axes `(time, body)` and trains on dimensionless standardized inputs/targets.
- `MPNN.predict`: accepts raw graph inputs in `[kg, m, m/s]`, applies `ScalerBundle`, runs `forward`, and returns acceleration in m/s^2.
- `MPNN.edge_messages`: expects already-scaled node inputs because PySR distills the internal learned message function, not the raw public prediction API.


In [ ]:
print_block("Horizons Source Units")
print(HORIZONS_UNITS)
print_block("Graph Artifact Units")
print(GRAPH_UNITS)

try:
    import pandas as pd
    horizons = pd.read_csv(Path(config.DATA_DIR) / "horizons.csv", nrows=5)
    print("\nHorizons CSV head:")
    display(horizons)
    jd_step_seconds = np.diff(horizons["JDTDB"].to_numpy()) * 86400
    print(f"cadence from JDTDB: {jd_step_seconds} seconds")
    print(f"first raw |r|: {np.linalg.norm(horizons.loc[0, ['X', 'Y', 'Z']].to_numpy(dtype=float)):.6g} km")
    print(f"first raw |v|: {np.linalg.norm(horizons.loc[0, ['VX', 'VY', 'VZ']].to_numpy(dtype=float)):.6g} km/s")
except Exception as exc:
    print(f"Horizons CSV inspection skipped: {exc}")


## 5. Optional: Archive Existing GNN Scaler

Training fits scalers from the graph data. The current datamodule only writes `artifacts/gnn_scaler.pkl` when that file does not already exist, so archive stale scalers before a new graph training attempt.

Leave `ARCHIVE_EXISTING_SCALER = False` when inspecting an existing trained model that already depends on the current scaler.


In [ ]:
ARCHIVE_EXISTING_SCALER = False

scaler_path = Path(config.SCALER_FILE)
if ARCHIVE_EXISTING_SCALER and scaler_path.exists():
    archive_dir = scaler_path.parent / "old"
    archive_dir.mkdir(parents=True, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    archived = archive_dir / f"{scaler_path.stem}.{stamp}{scaler_path.suffix}"
    shutil.move(str(scaler_path), str(archived))
    print(f"Archived scaler to {archived}")
elif scaler_path.exists():
    print(f"Scaler left in place: {scaler_path}")
else:
    print("No scaler file currently exists.")


## 6. Build Graph Snapshot Tensor

This creates `data/graph_data.npy` with shape `(T, B, 10)` and column order:

`[mass, x, y, z, vx, vy, vz, ax, ay, az]`

For the current default, this builds a Sun + Mercury graph from `data/horizons.csv`. Rebuild whenever the unit logic changes, the source CSV changes, or the existing physical sanity check reports acceleration off by a factor near `SECONDS_PER_JULIAN_YEAR`.


In [ ]:
FORCE_REBUILD_GRAPH = False
MERCURY_CSV = Path(config.DATA_DIR) / "horizons.csv"
GRAPH_FILE = Path(config.GRAPH_FILE)

if not MERCURY_CSV.exists():
    raise FileNotFoundError(
        f"Missing {MERCURY_CSV}. Run data_init.py or provide the intended Horizons CSV first."
    )

if FORCE_REBUILD_GRAPH or not GRAPH_FILE.exists():
    GRAPH_FILE.parent.mkdir(parents=True, exist_ok=True)
    snapshots = build_graph_snapshots(str(MERCURY_CSV), out_file=str(GRAPH_FILE))
    print(f"Built {GRAPH_FILE}: shape={snapshots.shape}, dtype={snapshots.dtype}")
else:
    snapshots = np.load(GRAPH_FILE, mmap_mode="r")
    print(f"Using existing {GRAPH_FILE}: shape={snapshots.shape}, dtype={snapshots.dtype}")


## 7. Validate Graph Artifact

These checks catch the common pre-training failures: wrong shape, non-finite data, missing masses, nonzero fixed-Sun state, or empty acceleration targets.


In [ ]:
graph = inspect_graph_file(config.GRAPH_FILE)

assert graph.ndim == 3, f"Expected [T, B, F], got shape {graph.shape}"
T, B, F = graph.shape
assert F == 10, f"Expected 10 columns [mass, xyz, vxyz, axyz], got {F}"
assert B >= 2, f"Expected at least Sun + Mercury, got B={B}"
assert finite_all(graph), "graph_data.npy contains non-finite values"
assert graph[0, 0, 0] > 1e29, "Sun mass is missing or implausible"
assert graph[0, 1, 0] > 1e20, "Mercury mass is missing or implausible"

sun_non_mass_max = np.max(np.abs(graph[:, 0, 1:]))
mercury_target_std = np.std(graph[:, 1, 7:10], axis=0)

assert sun_non_mass_max == 0.0, f"Sun row should be fixed at zero, got max {sun_non_mass_max:g}"
assert np.all(mercury_target_std > 0), "Mercury acceleration targets have zero variance"
print("Graph artifact checks passed.")


In [ ]:
# Quick physical sanity check: Mercury's target acceleration should agree with the
# Newtonian solar acceleration in order of magnitude for a Sun + Mercury graph.
graph = np.load(config.GRAPH_FILE, mmap_mode="r")
check_indices = [0, min(100, graph.shape[0] - 1), graph.shape[0] // 2]
ratios = []
for idx in check_indices:
    summary = print_physical_summary(graph, idx=idx)
    ratios.append(summary["target_to_newton_ratio"])

ratios = np.asarray(ratios)
assert np.all((0.95 < ratios) & (ratios < 1.05)), (
    "Graph acceleration targets are not in m/s^2 or are physically inconsistent. "
    f"target/newton ratios: {ratios}"
)
print("Graph target acceleration units look like m/s^2.")


## 8. Datamodule And Model Smoke Tests

This verifies that the graph artifact can feed the active `GraphDataset`, that batching produces the expected graph dictionary, and that an untrained `MPNN.forward` returns finite tensors with the right shape.


In [ ]:
print_block("Datamodule And Forward Smoke Test")
dm = NNDataModule(batch_size=2)
batch = next(iter(dm.train_dataloader()))
x, y = batch

model = MPNN()
with torch.no_grad():
    out = model(x)

print(f"nodes:        {tuple(x['nodes'].shape)}")
print(f"src_index:    {tuple(x['src_index'].shape)}")
print(f"dst_index:    {tuple(x['dst_index'].shape)}")
print(f"predict_mask: {x.get('predict_mask')}")
print(f"targets:      {tuple(y.shape)}")
print(f"forward out:  {tuple(out.shape)}")
describe_numpy("batch target sample", y.detach().cpu().numpy())

assert x['nodes'].shape[-1] == MPNN.input_dim
assert y.shape[-1] == MPNN.output_dim
assert out.shape == y.shape
assert torch.isfinite(out).all(), "MPNN.forward produced non-finite values"
print("Datamodule/model smoke test passed.")


## 9. Training Step And Gradient Smoke Test

This exercises the same scaled datamodule batch, masked loss, message L1 regularization path, and backward pass used by Lightning training. It does not update weights; it only confirms the training graph is finite and differentiable.


In [ ]:
print_block("Training Step / Backward Smoke Test")
model.train()
y_hat = model(x)
train_loss = model._masked_loss(y_hat, y.view_as(y_hat), x.get("predict_mask"))
if model.msg_l1 and model.last_messages is not None:
    train_loss = train_loss + model.msg_l1 * model.last_messages.abs().mean()

assert torch.isfinite(train_loss), f"non-finite training loss: {train_loss}"
train_loss.backward()

grad_sq = torch.tensor(0.0, dtype=torch.get_default_dtype())
n_grad = 0
for param in model.parameters():
    if param.grad is not None:
        assert torch.isfinite(param.grad).all(), "non-finite gradient detected"
        grad_sq = grad_sq + param.grad.detach().pow(2).sum().cpu()
        n_grad += 1
grad_norm = grad_sq.sqrt().item()
print(f"training loss: {train_loss.item():.6g}")
print(f"gradient tensors: {n_grad}")
print(f"gradient norm: {grad_norm:.6g}")
assert n_grad > 0, "no gradients were produced"
assert np.isfinite(grad_norm), "gradient norm is non-finite"
model.zero_grad(set_to_none=True)
model.eval()


## 10. Scaling And Public Prediction Invariant

`MPNN.predict` is the public raw-physics-unit API. It should match the explicit manual path:

`raw batch -> ScalerBundle.scale_graph_batch -> forward -> ScalerBundle.unscale_targets`

This test is independent of model quality; the model is still randomly initialized here unless you load a checkpoint below.


In [ ]:
raw_graph = np.asarray(np.load(config.GRAPH_FILE, mmap_mode="r")[:2]).copy()
num_bodies = raw_graph.shape[1]
src_index, dst_index = fully_connected_edges(num_bodies)

raw_batch = {
    "nodes": torch.from_numpy(raw_graph[..., MPNN.input_slice]).to(dtype=torch.get_default_dtype()),
    "src_index": src_index,
    "dst_index": dst_index,
    "predict_mask": torch.arange(1, num_bodies, dtype=torch.long),
}

model_for_invariant = globals().get("model")
if model_for_invariant is None:
    print("No existing `model` found from the smoke-test section; creating a fresh MPNN.")
    model_for_invariant = MPNN()

scalers = ScalerBundle.maybe_load()
if scalers is None:
    print("config.SCALE is False; predict should equal forward on raw inputs.")
    with torch.no_grad():
        pred_public = model_for_invariant.predict(raw_batch)
        pred_manual = model_for_invariant(raw_batch)
else:
    with torch.no_grad():
        pred_public = model_for_invariant.predict(raw_batch)
        scaled_batch = scalers.scale_graph_batch(raw_batch)
        pred_manual = scalers.unscale_targets(model_for_invariant(scaled_batch))

max_abs_diff = torch.max(torch.abs(pred_public - pred_manual)).item()
print(f"public prediction shape: {tuple(pred_public.shape)}")
print(f"max |predict - manual|: {max_abs_diff:.6g}")
assert torch.isfinite(pred_public).all(), "MPNN.predict produced non-finite values"
assert torch.allclose(pred_public, pred_manual, rtol=1e-5, atol=1e-8), "predict scaling invariant failed"


## 11. Edge Message And PySR Interface Smoke Test

`edge_messages` is intentionally an internal scaled-space interface. This checks that the edge inputs and messages are finite and have the expected dimensions before a PySR distillation run.


In [ ]:
print_block("Edge Message / PySR Interface Smoke Test")
scalers = ScalerBundle.maybe_load()
if scalers is not None:
    edge_batch = scalers.scale_graph_batch(raw_batch)
else:
    edge_batch = raw_batch

edge_input, messages = model_for_invariant.edge_messages(edge_batch)
print(f"edge_input shape: {edge_input.shape}")
print(f"messages shape:   {messages.shape}")
assert edge_input.shape == (raw_batch["nodes"].shape[0], src_index.numel(), 2 * MPNN.input_dim)
assert messages.shape == (raw_batch["nodes"].shape[0], src_index.numel(), model_for_invariant.msg_dim)
assert np.isfinite(edge_input).all(), "non-finite edge inputs"
assert np.isfinite(messages).all(), "non-finite edge messages"


## 12. Integration Force Wrapper Smoke Test

This checks that the orbit integrator's learned-force path builds a raw Sun + Mercury graph and receives finite physical-unit accelerations from `MPNN.predict`.


In [ ]:
print_block("Integration Force Wrapper Smoke Test")
sample = np.asarray(np.load(config.GRAPH_FILE, mmap_mode="r")[0]).copy()
r_vec = sample[1, 1:4]
v_vec = sample[1, 4:7]
a_newton = newton_force(r_vec)
a_gnn = gnn_force(model_for_invariant, r_vec, v_vec)
print(f"Newton |a|: {np.linalg.norm(a_newton):.6g} m/s^2")
print(f"GNN wrapper prediction shape: {a_gnn.shape}")
print(f"GNN wrapper finite: {np.isfinite(a_gnn).all()}")
assert a_gnn.shape == (3,)
assert np.isfinite(a_gnn).all(), "gnn_force returned non-finite acceleration"


## 13. Optional: Checkpoint Evaluation Smoke Test

Use this after a training job has produced a checkpoint. Keep inference through `model.predict` for raw graph batches.


In [ ]:
print_block("Checkpoint Compatibility Scan")
compatible_ckpts = compatible_mpnn_checkpoints()
CKPT = compatible_ckpts[0] if compatible_ckpts else None

if CKPT is not None:
    print_block(f"Checkpoint Smoke Test: {CKPT}")
    trained = load_current_mpnn_checkpoint(CKPT)
    trained.eval()
    with torch.no_grad():
        pred = trained.predict(raw_batch)

    target = torch.from_numpy(raw_graph[..., MPNN.output_slice].copy()).to(pred.device, dtype=pred.dtype)
    mask = raw_batch["predict_mask"].to(pred.device)
    mae = calc_mae(pred[:, mask, :], target[:, mask, :])
    mape = calc_mape(pred[:, mask, :], target[:, mask, :])

    print(f"prediction shape: {tuple(pred.shape)}")
    print(f"finite: {torch.isfinite(pred).all().item()}")
    print(f"Mercury acceleration prediction[0]: {pred[0, 1].detach().cpu().numpy()}")
    print(f"small-batch MAE: {mae:.6g}")
    print(f"small-batch MAPE: {mape:.6g}%")

    ntrials = min(256, np.load(config.GRAPH_FILE, mmap_mode="r").shape[0])
    gnn_test(trained, ntrials=ntrials, batch_size=32, mape=True, suppress=False, mean_axis=None)
else:
    print("No compatible current-MPNN checkpoint found. Legacy flat checkpoints were skipped as expected.")


## 14. Optional: Optuna Study Inspection

Use the repo's `utils.optuna_helpers` for quick summaries and visual diagnostics. The helper expects the study name to match the DB basename, so this cell runs it from the DB directory with the basename. It also keeps a direct `optuna.load_study` fallback for richer dataframe inspection.


In [ ]:
try:
    import optuna
    import optuna.visualization as optuna_vis

    study_paths = sorted(Path("artifacts").glob("*study.db")) + sorted(Path(".").glob("*study.db"))
    if not study_paths:
        print("No Optuna study DBs found.")
    for study_path in study_paths:
        print_block(f"Optuna helper summary: {study_path}")
        try:
            with pushd(study_path.parent):
                print_best_optuna(study_path.name)
                USE_OPTUNA_HELPER_PLOTS = False
                if USE_OPTUNA_HELPER_PLOTS:
                    plot_pareto(study_path.name, plot_param_importances=True)
        except Exception as helper_exc:
            print(f"optuna_helpers summary/plots skipped: {helper_exc}")

        study_name = study_path.stem
        storage = f"sqlite:///{study_path}"
        try:
            study = optuna.load_study(study_name=study_name, storage=storage)
            print(f"loaded study: {study.study_name}")
            print(f"trial count: {len(study.trials)}")
            print(f"best value: {study.best_value}")
            if study_looks_current_gnn(study):
                print("Study appears to contain current MPNN/GNN hyperparameters.")
            else:
                print("WARNING: study does not contain current GNN params; treat it as legacy guidance only.")
            print("best params:")
            for key, value in study.best_params.items():
                print(f"  {key}: {value}")

            trials_df = study.trials_dataframe(attrs=("number", "value", "state", "params"))
            display(trials_df.sort_values("value").head(10))

            if study_looks_current_gnn(study):
                optuna_vis.plot_optimization_history(study).show()
                optuna_vis.plot_param_importances(study).show()
        except Exception as direct_exc:
            print(f"Direct Optuna inspection skipped for {study_path}: {direct_exc}")
except Exception as exc:
    print(f"Optuna inspection skipped: {exc}")


## 15. Optional: PySR Distillation Artifact Inspection

PySR now distills the learned internal edge/message function. Message inputs are intentionally in the scaled node space used by `phi_e`, not the public raw-unit prediction API.


In [ ]:
try:
    import joblib

    distilled_path = Path("pysr/distilled.pkl")
    if distilled_path.exists():
        distilled = joblib.load(distilled_path)
        print(f"active channels: {distilled.get('active_channels')}")
        print(f"variable names: {distilled.get('variable_names')}")
        print("edge expressions:")
        for channel, expr in distilled.get("edge_expressions", {}).items():
            print(f"  channel {channel}: {expr}")
    else:
        print(f"No distilled PySR artifact found at {distilled_path}")
except Exception as exc:
    print(f"PySR artifact inspection skipped: {exc}")


## 16. Training Submit Handoff

This notebook is ready when the graph build, unit checks, datamodule smoke test, backward-pass test, prediction invariant, edge-message test, and integration wrapper test all pass.

Before submitting the first real training job:

1. Confirm `data/graph_data.npy` was rebuilt after the unit fix and passes the target/Newton ratio checks.
2. Confirm `artifacts/gnn_scaler.pkl` was generated from the corrected graph data.
3. Treat existing checkpoints and old Optuna studies as legacy unless the compatibility scans identify current-MPNN artifacts.
4. Decide whether the cluster run should keep `config.MAC = True` as a small float32/dev profile or switch it to `False` for full cluster settings.
5. Submit from a shell with `sbatch scripts/run_train.sh`, then watch the first log page for environment, graph, scaler, CUDA/DDP, OOM, and NaN-loss failures.
